In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report
)

In [ ]:
df = pd.read_csv("clean_credit_risk.csv")

df.head()

In [ ]:
df.info()

In [ ]:
X = df.drop("loan_status", axis=1)

y = df["loan_status"]

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
y.value_counts()

In [ ]:
sns.countplot(x=y)

plt.title("Loan Default Distribution")
plt.xlabel("Default Status")
plt.ylabel("Count")

plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
df = pd.read_csv("credit_processed.csv")

# Encode categorical variables
df = pd.get_dummies(
    df,
    drop_first=True
)

df.head()

In [ ]:
df.dtypes

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop("loan_status", axis=1)
y = df["loan_status"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
log_predictions = log_model.predict(X_test_scaled)

log_probabilities = log_model.predict_proba(
    X_test_scaled
)[:,1]

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(
    X_train,
    y_train
)

In [ ]:
rf_predictions = rf_model.predict(X_test)

rf_probabilities = rf_model.predict_proba(
    X_test
)[:,1]

In [ ]:
def evaluate_model(name, y_true, predictions, probabilities):

    print("="*50)
    print(name)
    print("="*50)

    print(
        classification_report(
            y_true,
            predictions
        )
    )

    print(
        "Accuracy:",
        round(
            accuracy_score(y_true, predictions),
            3
        )
    )

    print(
        "Precision:",
        round(
            precision_score(y_true, predictions),
            3
        )
    )

    print(
        "Recall:",
        round(
            recall_score(y_true, predictions),
            3
        )
    )

    print(
        "F1 Score:",
        round(
            f1_score(y_true, predictions),
            3
        )
    )

    print(
        "ROC-AUC:",
        round(
            roc_auc_score(y_true, probabilities),
            3
        )
    )

In [ ]:
evaluate_model(
    "Logistic Regression",
    y_test,
    log_predictions,
    log_probabilities
)

In [ ]:
evaluate_model(
    "Random Forest",
    y_test,
    rf_predictions,
    rf_probabilities
)

In [ ]:
cm_log = confusion_matrix(
    y_test,
    log_predictions
)

sns.heatmap(
    cm_log,
    annot=True,
    fmt="d"
)

plt.title(
    "Logistic Regression Confusion Matrix"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()

In [ ]:
plt.figure(figsize=(8,6))


RocCurveDisplay.from_predictions(
    y_test,
    log_probabilities,
    name="Logistic Regression"
)


RocCurveDisplay.from_predictions(
    y_test,
    rf_probabilities,
    name="Random Forest"
)


plt.title(
    "ROC Curve Comparison"
)

plt.show()

In [ ]:
results = pd.DataFrame({

    "Model":
    [
        "Logistic Regression",
        "Random Forest"
    ],

    "Accuracy":
    [
        accuracy_score(y_test, log_predictions),
        accuracy_score(y_test, rf_predictions)
    ],

    "Precision":
    [
        precision_score(y_test, log_predictions),
        precision_score(y_test, rf_predictions)
    ],

    "Recall":
    [
        recall_score(y_test, log_predictions),
        recall_score(y_test, rf_predictions)
    ],

    "F1 Score":
    [
        f1_score(y_test, log_predictions),
        f1_score(y_test, rf_predictions)
    ],

    "ROC-AUC":
    [
        roc_auc_score(y_test, log_probabilities),
        roc_auc_score(y_test, rf_probabilities)
    ]

})


results.round(3)

In [ ]:
feature_importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance":
    rf_model.feature_importances_

})


feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)


feature_importance.head(15)

In [ ]:
plt.figure(figsize=(10,6))

sns.barplot(
    data=feature_importance.head(15),
    x="Importance",
    y="Feature"
)


plt.title(
    "Top 15 Random Forest Feature Importance"
)

plt.show()

In [ ]:
import joblib

joblib.dump(rf_model, "best_credit_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Model and scaler saved successfully!")

# Model Evaluation and Comparison

## Overview

Two machine learning models were developed to predict loan default risk:

1. **Logistic Regression**
   - Used as the baseline model.
   - Selected because it is widely used in credit risk applications due to its interpretability and transparency.

2. **Random Forest**
   - Used as a more flexible ensemble model.
   - Able to capture nonlinear relationships and interactions between borrower characteristics.

Both models were evaluated using:
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion Matrix
- ROC Curve
- Feature Importance

---

# Logistic Regression Results

The Logistic Regression model achieved the following performance:

| Metric | Score |
|---|---:|
| Accuracy | 0.869 |
| Precision | 0.772 |
| Recall | 0.558 |
| F1-score | 0.648 |
| ROC-AUC | 0.874 |

### Classification Report

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| 0 (Non-default) | 0.89 | 0.95 | 0.92 | 4943 |
| 1 (Default) | 0.77 | 0.56 | 0.65 | 1362 |

The Logistic Regression model performs well at identifying non-default customers, achieving a high recall of 0.95 for class 0. However, its ability to correctly identify default cases is lower, with a recall of 0.56.

The ROC-AUC score of 0.874 indicates that the model has good ability to distinguish between default and non-default customers.

---

# Random Forest Results

The Random Forest model achieved the following performance:

| Metric | Score |
|---|---:|
| Accuracy | 0.935 |
| Precision | 0.962 |
| Recall | 0.729 |
| F1-score | 0.830 |
| ROC-AUC | 0.935 |

### Classification Report

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
| 0 (Non-default) | 0.93 | 0.99 | 0.96 | 4943 |
| 1 (Default) | 0.96 | 0.73 | 0.83 | 1362 |

The Random Forest model significantly improves prediction performance compared with Logistic Regression.

It achieves:
- Higher accuracy (93.5%)
- Higher precision (96.2%)
- Higher recall for default customers (72.9%)
- Higher F1-score (83.0%)
- Higher ROC-AUC (93.5%)

The improved recall is particularly important for credit risk because correctly identifying potential defaulters helps reduce financial losses.

---

# Model Comparison

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.869 | 0.772 | 0.558 | 0.648 | 0.874 |
| Random Forest | 0.935 | 0.962 | 0.729 | 0.830 | 0.935 |

The Random Forest model outperformed Logistic Regression across all evaluation metrics.

---

# Confusion Matrix Analysis

## Logistic Regression

The confusion matrix showed:

- True Negatives: 4719
- False Positives: 224
- False Negatives: 602
- True Positives: 760

The model correctly identified many non-default customers but missed a considerable number of actual defaults.

---

## Random Forest

The Random Forest confusion matrix demonstrated improved classification performance, particularly for default detection.

The model reduced false negatives compared with Logistic Regression, meaning fewer risky customers were incorrectly classified as safe.

---

# ROC Curve Analysis

The ROC curves demonstrate the difference in classification ability:

- Logistic Regression ROC-AUC: **0.874**
- Random Forest ROC-AUC: **0.935**

The Random Forest model provides stronger separation between default and non-default customers.

---

# Random Forest Feature Importance

The Random Forest model identified the following top predictors of loan default:

| Rank | Feature | Importance |
|---|---|---:|
| 1 | loan_percent_income | 0.220 |
| 2 | person_income | 0.145 |
| 3 | loan_int_rate | 0.122 |
| 4 | loan_amnt | 0.079 |
| 5 | person_home_ownership_RENT | 0.075 |
| 6 | person_emp_length | 0.062 |
| 7 | loan_grade_D | 0.057 |
| 8 | person_age | 0.050 |
| 9 | cb_person_cred_hist_length | 0.038 |
| 10 | loan_grade_C | 0.018 |
| 11 | loan_intent_MEDICAL | 0.017 |
| 12 | loan_grade_E | 0.016 |
| 13 | person_home_ownership_OWN | 0.016 |
| 14 | loan_intent_HOMEIMPROVEMENT | 0.016 |
| 15 | loan_intent_EDUCATION | 0.015 |

The most influential variable was **loan_percent_income**, indicating that the borrower's loan burden relative to income is a strong predictor of default risk.

Other important factors included:
- Borrower income
- Interest rate
- Loan amount
- Employment length
- Loan grade
- Housing status

---

# Final Model Selection

Although Logistic Regression provides better interpretability and is commonly used in regulated credit risk environments, the Random Forest model achieved substantially better predictive performance.

The Random Forest model is preferred for this project because it:

- Detects more default cases
- Achieves higher ROC-AUC
- Produces stronger overall classification performance
- Identifies important risk factors through feature importance analysis

However, in real-world banking applications, the final model choice would depend on the balance between predictive accuracy and explainability. Logistic Regression may still be preferred where strict regulatory transparency is required, while Random Forest may be preferred when maximizing predictive accuracy is the priority.

In [ ]:
import joblib

joblib.dump(rf_model, "best_credit_model.pkl")

print("Random Forest model saved successfully!")